# Construction Site VLM Dataset - Validation
Loads the cleaned dataset saved by the `01_dataset_exploration.ipynb` notebook, verifies all fixes were applied correctly, and regenerates the full stats / co-occurrence CSVs on the cleaned data before fine-tuning begins.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

import os
import json
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from datasets import load_from_disk
from collections import Counter
from tqdm.auto import tqdm

CLEANED_DATASET_PATH = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/raw_cleaned"
VALIDATION_SAVE_DIR = "/content/drive/MyDrive/vlm-finetuning-project1/datasets/stats/cleaned"
os.makedirs(VALIDATION_SAVE_DIR, exist_ok=True)

check_results = []  # collects (check_name, passed: bool, detail: str)


def record(name, passed, detail=""):
    check_results.append({"check": name, "passed": bool(passed), "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""))

In [2]:
dataset = load_from_disk(CLEANED_DATASET_PATH)
print(dataset)

EXPECTED_COUNTS = {"train": 7009, "test": 3004}
for split, expected in EXPECTED_COUNTS.items():
    actual = len(dataset[split])
    record(f"split_size_{split}", actual == expected,
           f"expected {expected}, got {actual}")

DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat'],
        num_rows: 7009
    })
    test: Dataset({
        features: ['image', 'image_id', 'image_caption', 'illumination', 'camera_distance', 'view', 'quality_of_info', 'rule_1_violation', 'rule_2_violation', 'rule_3_violation', 'rule_4_violation', 'excavator', 'rebar', 'worker_with_white_hard_hat'],
        num_rows: 3004
    })
})
[PASS] split_size_train — expected 7009, got 7009
[PASS] split_size_test — expected 3004, got 3004


## Schema & Basic Integrity Check

In [3]:
EXPECTED_KEYS = {
    "image", "image_id", "image_caption", "illumination", "camera_distance",
    "view", "quality_of_info", "rule_1_violation", "rule_2_violation",
    "rule_3_violation", "rule_4_violation", "excavator", "rebar",
    "worker_with_white_hard_hat",
}

for split_name in ["train", "test"]:
    sample_keys = set(dataset[split_name][0].keys())
    missing_keys = EXPECTED_KEYS - sample_keys
    record(f"schema_keys_{split_name}", not missing_keys, f"missing: {missing_keys or 'none'}")

    ids = [s["image_id"] for s in dataset[split_name]]
    n_unique = len(set(ids))
    record(f"unique_image_ids_{split_name}", n_unique == len(ids),
           f"{len(ids) - n_unique} duplicate IDs found")

[PASS] schema_keys_train — missing: none
[PASS] unique_image_ids_train — 0 duplicate IDs found
[PASS] schema_keys_test — missing: none
[PASS] unique_image_ids_test — 0 duplicate IDs found


## Bounding Box Validation (Re-run)

After fixes, this should report **zero** invalid or missing boxes.

In [4]:
def validate_box(box, tol=1e-6):
    xmin, ymin, xmax, ymax = box
    issues = []
    if not all(0 - tol <= c <= 1 + tol for c in box):
        issues.append("out_of_range")
    if xmin >= xmax - tol:
        issues.append("invalid_x_order_or_zero_width")
    if ymin >= ymax - tol:
        issues.append("invalid_y_order_or_zero_height")
    area = max(0, xmax - xmin) * max(0, ymax - ymin)
    if area < 1e-5:
        issues.append("near_zero_area")
    return issues, area


def audit_boxes(split, split_name):
    records_ = []
    for sample in split:
        image_id = sample["image_id"]
        objects_to_check = {
            "excavator": sample.get("excavator", []),
            "rebar": sample.get("rebar", []),
            "white_hard_hat": sample.get("worker_with_white_hard_hat", []),
        }
        for cls_name, boxes in objects_to_check.items():
            for box in boxes:
                issues, area = validate_box(box)
                records_.append({"split": split_name, "image_id": image_id, "class": cls_name,
                                  "box": box, "area": area, "issues": ";".join(issues) or "ok"})

        for i in range(1, 5):
            v = sample.get(f"rule_{i}_violation")
            if v is None:
                continue
            boxes = v.get("bounding_box")
            if not boxes:
                records_.append({"split": split_name, "image_id": image_id,
                                  "class": f"rule_{i}_violation", "box": [], "area": 0.0,
                                  "issues": "missing_box"})
            else:
                for box in boxes:
                    issues, area = validate_box(box)
                    records_.append({"split": split_name, "image_id": image_id,
                                      "class": f"rule_{i}_violation", "box": box, "area": area,
                                      "issues": ";".join(issues) or "ok"})
    return pd.DataFrame(records_)


df_boxes_train = audit_boxes(dataset["train"], "train")
df_boxes_test = audit_boxes(dataset["test"], "test")
df_boxes_all = pd.concat([df_boxes_train, df_boxes_test], ignore_index=True)

invalid_boxes = df_boxes_all[df_boxes_all["issues"] != "ok"].copy()
record("no_invalid_or_missing_boxes", len(invalid_boxes) == 0,
       f"{len(invalid_boxes)} invalid/missing boxes remain")

if len(invalid_boxes):
    display(invalid_boxes.groupby(["split", "class", "issues"]).size().unstack(fill_value=0))
invalid_boxes.to_csv(f"{VALIDATION_SAVE_DIR}/remaining_invalid_boxes.csv", index=False)

[PASS] no_invalid_or_missing_boxes — 0 invalid/missing boxes remain


## Verify Fixes

In [5]:
def find_sample_by_image_id(split, image_id):
    for sample in split:
        if str(sample["image_id"]) == str(image_id):
            return sample
    return None


fix_checks = [
    ("train", "0005014", "excludes_box",
     lambda s: [0.68, 0.19, 0.68, 0.2] not in (s["rule_1_violation"]["bounding_box"] if s["rule_1_violation"] else [])),
    ("train", "0006857", "excludes_box",
     lambda s: [0.95, 0.45, 0.95, 0.46] not in s["worker_with_white_hard_hat"]),
    ("train", "0011654", "excludes_box",
     lambda s: [0.93, 0.95, 0.94, 0.95] not in s["excavator"]),
    ("train", "0013188", "excludes_box",
     lambda s: [0.9, 0.43, 0.91, 0.43] not in s["rebar"]),
    ("train", "0009782", "box_replaced",
     lambda s: s["rule_1_violation"] is not None and s["rule_1_violation"]["bounding_box"] == [[0.08, 0.65, 0.14, 0.75]]),
    ("test", "0000167", "violation_removed",
     lambda s: s["rule_1_violation"] is None),
    ("test", "0000388", "camera_distance_set",
     lambda s: s["camera_distance"] == "mid distance"),
]

for split_name, image_id, fix_type, check_fn in fix_checks:
    sample = find_sample_by_image_id(dataset[split_name], image_id)
    if sample is None:
        record(f"fix_{image_id}_{fix_type}", False, "sample not found in dataset")
        continue
    passed = check_fn(sample)
    record(f"fix_{image_id}_{fix_type}", passed)

[PASS] fix_0005014_excludes_box
[PASS] fix_0006857_excludes_box
[PASS] fix_0011654_excludes_box
[PASS] fix_0013188_excludes_box
[PASS] fix_0009782_box_replaced
[PASS] fix_0000167_violation_removed
[PASS] fix_0000388_camera_distance_set


## Recompute Dataset Statistics

In [6]:
def extract_and_save_stats(split_data, split_name, save_directory):
    print(f"Crunching statistics for the {split_name.upper()} split...")

    stats = {
        "Illumination": Counter(), "Camera Distance": Counter(),
        "View": Counter(), "Quality of Info": Counter(),
        "Rule Violations": Counter(), "Object Presence": Counter(),
    }

    for sample in tqdm(split_data, desc=f"{split_name.capitalize()} Progress"):
        stats["Illumination"][sample["illumination"]] += 1
        stats["Camera Distance"][sample["camera_distance"]] += 1
        stats["View"][sample["view"]] += 1
        stats["Quality of Info"][sample["quality_of_info"]] += 1

        rule_labels = {1: "Rule 1 (PPE)", 2: "Rule 2 (Harness)",
                        3: "Rule 3 (Edge Protection)", 4: "Rule 4 (Blind Spot)"}
        for i, label in rule_labels.items():
            if sample[f"rule_{i}_violation"]:
                stats["Rule Violations"][label] += 1

        if sample["excavator"]:
            stats["Object Presence"]["Excavator"] += 1
        if sample["rebar"]:
            stats["Object Presence"]["Rebar"] += 1
        if sample["worker_with_white_hard_hat"]:
            stats["Object Presence"]["White Hard Hat"] += 1

    flat_data = [
        {"Category": category, "Attribute": str(attribute), "Count": count}
        for category, counter in stats.items()
        for attribute, count in counter.items()
    ]

    df_stats = pd.DataFrame(flat_data).sort_values(
        by=["Category", "Count"], ascending=[True, False]
    ).reset_index(drop=True)

    print(f"\n--- {split_name.upper()} SPLIT STATISTICS (CLEANED) ---")
    display(df_stats)

    os.makedirs(save_directory, exist_ok=True)
    out_path = os.path.join(save_directory, f"construction_site_{split_name}_stats.csv")
    df_stats.to_csv(out_path, index=False)
    print(f"[{split_name.upper()}] Statistics saved to:\n{out_path}\n")

    return df_stats


df_train_stats = extract_and_save_stats(dataset["train"], "train", VALIDATION_SAVE_DIR)
df_test_stats = extract_and_save_stats(dataset["test"], "test", VALIDATION_SAVE_DIR)

# Confirm the None row is gone from Camera Distance
n_none_dist = df_test_stats[
    (df_test_stats["Category"] == "Camera Distance") & (df_test_stats["Attribute"] == "None")
].shape[0]
record("no_none_camera_distance_in_stats", n_none_dist == 0,
       f"{n_none_dist} 'None' row(s) still present in test Camera Distance stats")

Crunching statistics for the TRAIN split...


Train Progress:   0%|          | 0/7009 [00:00<?, ?it/s]


--- TRAIN SPLIT STATISTICS (CLEANED) ---


,Category,Attribute,Count
0,Camera Distance,mid distance,5063
1,Camera Distance,short distance,1560
2,Camera Distance,long distance,386
3,Illumination,normal lighting,5885
4,Illumination,underexposed,692
5,Illumination,overexposed,273
6,Illumination,night,159
7,Object Presence,Excavator,2415
8,Object Presence,Rebar,846
9,Object Presence,White Hard Hat,680


[TRAIN] Statistics saved to:
/content/drive/MyDrive/vlm-finetuning-project1/datasets/stats/cleaned/construction_site_train_stats.csv

Crunching statistics for the TEST split...


Test Progress:   0%|          | 0/3004 [00:00<?, ?it/s]


--- TEST SPLIT STATISTICS (CLEANED) ---


,Category,Attribute,Count
0,Camera Distance,short distance,1360
1,Camera Distance,mid distance,1310
2,Camera Distance,long distance,334
3,Illumination,normal lighting,2426
4,Illumination,underexposed,381
5,Illumination,overexposed,154
6,Illumination,night,43
7,Object Presence,Excavator,1080
8,Object Presence,Rebar,327
9,Object Presence,White Hard Hat,314


[TEST] Statistics saved to:
/content/drive/MyDrive/vlm-finetuning-project1/datasets/stats/cleaned/construction_site_test_stats.csv

[PASS] no_none_camera_distance_in_stats — 0 'None' row(s) still present in test Camera Distance stats


## Co-occurrence Analysis & Cross-check vs Paper (re-run on cleaned data)

In [7]:
def compute_cooccurrence(split, split_name):
    n = len(split)
    rule_flags = {f"rule_{i}": [] for i in range(1, 5)}
    obj_flags = {"excavator": [], "rebar": [], "white_hard_hat": []}

    for sample in split:
        for i in range(1, 5):
            rule_flags[f"rule_{i}"].append(1 if sample[f"rule_{i}_violation"] else 0)
        obj_flags["excavator"].append(1 if sample["excavator"] else 0)
        obj_flags["rebar"].append(1 if sample["rebar"] else 0)
        obj_flags["white_hard_hat"].append(1 if sample["worker_with_white_hard_hat"] else 0)

    df_rules = pd.DataFrame(rule_flags)
    df_objs = pd.DataFrame(obj_flags)

    rate_summary = pd.concat([
        df_rules.mean().rename("rate") * 100,
        df_objs.mean().rename("rate") * 100,
    ]).to_frame()
    rate_summary["count"] = pd.concat([df_rules.sum(), df_objs.sum()]).astype(int)
    rate_summary["split"] = split_name
    rate_summary["n_images"] = n

    return rate_summary, df_rules, df_objs


train_rates, train_rules, train_objs = compute_cooccurrence(dataset["train"], "train")
test_rates, test_rules, test_objs = compute_cooccurrence(dataset["test"], "test")

print("=== Rate Summary (Train, cleaned) ===")
display(train_rates)
print("\n=== Rate Summary (Test, cleaned) ===")
display(test_rates)

train_rates.to_csv(f"{VALIDATION_SAVE_DIR}/cooccurrence_rates_train.csv")
test_rates.to_csv(f"{VALIDATION_SAVE_DIR}/cooccurrence_rates_test.csv")

=== Rate Summary (Train, cleaned) ===


,rate,count,split,n_images
rule_1,9.659010,677,train,7009
rule_2,0.841775,59,train,7009
rule_3,1.555143,109,train,7009
rule_4,0.656299,46,train,7009
excavator,34.455700,2415,train,7009
rebar,12.070195,846,train,7009
white_hard_hat,9.701812,680,train,7009



=== Rate Summary (Test, cleaned) ===


,rate,count,split,n_images
rule_1,10.752330,323,test,3004
rule_2,0.832224,25,test,3004
rule_3,2.097204,63,test,3004
rule_4,0.798935,24,test,3004
excavator,35.952064,1080,test,3004
rebar,10.885486,327,test,3004
white_hard_hat,10.452730,314,test,3004


In [8]:
train_rule_matrix = train_rules.T.dot(train_rules)
train_obj_matrix = train_objs.T.dot(train_objs)
test_rule_matrix = test_rules.T.dot(test_rules)
test_obj_matrix = test_objs.T.dot(test_objs)

print("=== Train Rule Co-occurrence Matrix (cleaned) ===")
display(train_rule_matrix)
print("=== Train Object Co-occurrence Matrix (cleaned) ===")
display(train_obj_matrix)
print("=== Test Rule Co-occurrence Matrix (cleaned) ===")
display(test_rule_matrix)
print("=== Test Object Co-occurrence Matrix (cleaned) ===")
display(test_obj_matrix)

train_rule_matrix.to_csv(f"{VALIDATION_SAVE_DIR}/rule_cooc_matrix_train.csv")
train_obj_matrix.to_csv(f"{VALIDATION_SAVE_DIR}/obj_cooc_matrix_train.csv")
test_rule_matrix.to_csv(f"{VALIDATION_SAVE_DIR}/rule_cooc_matrix_test.csv")
test_obj_matrix.to_csv(f"{VALIDATION_SAVE_DIR}/obj_cooc_matrix_test.csv")

print(f"\nSaved all co-occurrence stats to {VALIDATION_SAVE_DIR}")

=== Train Rule Co-occurrence Matrix (cleaned) ===


,rule_1,rule_2,rule_3,rule_4
rule_1,677,6,7,11
rule_2,6,59,0,0
rule_3,7,0,109,1
rule_4,11,0,1,46


=== Train Object Co-occurrence Matrix (cleaned) ===


,excavator,rebar,white_hard_hat
excavator,2415,135,219
rebar,135,846,97
white_hard_hat,219,97,680


=== Test Rule Co-occurrence Matrix (cleaned) ===


,rule_1,rule_2,rule_3,rule_4
rule_1,323,5,7,10
rule_2,5,25,2,0
rule_3,7,2,63,2
rule_4,10,0,2,24


=== Test Object Co-occurrence Matrix (cleaned) ===


,excavator,rebar,white_hard_hat
excavator,1080,49,97
rebar,49,327,32
white_hard_hat,97,32,314



Saved all co-occurrence stats to /content/drive/MyDrive/vlm-finetuning-project1/datasets/stats/cleaned


## Cross-check Cleaned Counts vs Paper Table 4

In [9]:
paper_train_counts = {
    "rule_1": 677, "rule_2": 59, "rule_3": 109, "rule_4": 46,
    "excavator": 2415, "rebar": 846, "white_hard_hat": 680,
}
paper_test_counts = {
    "rule_1": 323, "rule_2": 25, "rule_3": 63, "rule_4": 24,
    "excavator": 1080, "rebar": 327, "white_hard_hat": 314,
}


def cross_check(rates_df, paper_counts, split_label):
    print(f"\n=== Cross-check vs Paper Table 4 ({split_label} Split, cleaned) ===")
    for k, expected in paper_counts.items():
        actual = rates_df.loc[k, "count"] if k in rates_df.index else None
        status = "MATCH" if actual == expected else "DIFFERS (expected post-fix)"
        print(f"{k.ljust(15)} | Paper: {str(expected).ljust(4)} | Computed: {str(actual).ljust(4)} | {status}")


cross_check(train_rates, paper_train_counts, "Train")
cross_check(test_rates, paper_test_counts, "Test")


=== Cross-check vs Paper Table 4 (Train Split, cleaned) ===
rule_1          | Paper: 677  | Computed: 677  | MATCH
rule_2          | Paper: 59   | Computed: 59   | MATCH
rule_3          | Paper: 109  | Computed: 109  | MATCH
rule_4          | Paper: 46   | Computed: 46   | MATCH
excavator       | Paper: 2415 | Computed: 2415 | MATCH
rebar           | Paper: 846  | Computed: 846  | MATCH
white_hard_hat  | Paper: 680  | Computed: 680  | MATCH

=== Cross-check vs Paper Table 4 (Test Split, cleaned) ===
rule_1          | Paper: 323  | Computed: 323  | MATCH
rule_2          | Paper: 25   | Computed: 25   | MATCH
rule_3          | Paper: 63   | Computed: 63   | MATCH
rule_4          | Paper: 24   | Computed: 24   | MATCH
excavator       | Paper: 1080 | Computed: 1080 | MATCH
rebar           | Paper: 327  | Computed: 327  | MATCH
white_hard_hat  | Paper: 314  | Computed: 314  | MATCH


## Visual Check of Fixed Images

In [10]:
def _normalize_boxes(boxes):
    if not boxes:
        return []
    if isinstance(boxes[0], (int, float)):
        return [boxes]
    return boxes


def plot_sample(sample, show_objects=True, show_violations=True):
    img = sample["image"]
    width, height = img.size
    fig, ax = plt.subplots(1, figsize=(9, 7))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"ID: {sample['image_id']}")

    def draw_boxes(boxes, color, label):
        for box in _normalize_boxes(boxes):
            xmin, ymin, xmax, ymax = box
            xmin_px, xmax_px = xmin * width, xmax * width
            ymin_px, ymax_px = ymin * height, ymax * height
            box_width, box_height = xmax_px - xmin_px, ymax_px - ymin_px

            if box_width <= 0.1 or box_height <= 0.1:
                ax.plot([xmin_px, xmax_px], [ymin_px, ymax_px],
                        color="red", marker="o", markersize=10, linewidth=2)
                ax.text(xmin_px, ymin_px - 8, f"BROKEN (Line): {label}", color="white",
                        fontsize=9, fontweight="bold",
                        bbox=dict(facecolor="red", alpha=0.9, edgecolor="none", pad=2))
                return

            rect = patches.Rectangle((xmin_px, ymin_px), box_width, box_height,
                                      linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(xmin_px, ymin_px - 5, label, color="white", fontsize=9,
                    bbox=dict(facecolor=color, alpha=0.8, edgecolor="none", pad=1))

    if show_objects:
        draw_boxes(sample.get("excavator", []), "blue", "Excavator")
        draw_boxes(sample.get("rebar", []), "orange", "Rebar")
        draw_boxes(sample.get("worker_with_white_hard_hat", []), "green", "White Hard Hat")
    if show_violations:
        for i in range(1, 5):
            v = sample.get(f"rule_{i}_violation")
            if v and v.get("bounding_box"):
                draw_boxes(v["bounding_box"], "red", f"Rule {i} Violation")

    plt.tight_layout()
    plt.show()


for split_name, image_id, _, _ in fix_checks:
    sample = find_sample_by_image_id(dataset[split_name], image_id)
    if sample:
        plot_sample(sample)

Output hidden; open in https://colab.research.google.com to view.

## Report

In [11]:
report_df = pd.DataFrame(check_results)
report_path = f"{VALIDATION_SAVE_DIR}/validation_report.csv"
report_df.to_csv(report_path, index=False)

n_failed = (~report_df["passed"]).sum()
print(f"\nValidation report saved to: {report_path}")
display(report_df)


Validation report saved to: /content/drive/MyDrive/vlm-finetuning-project1/datasets/stats/cleaned/validation_report.csv


,check,passed,detail
0,split_size_train,True,"expected 7009, got 7009"
1,split_size_test,True,"expected 3004, got 3004"
2,schema_keys_train,True,missing: none
3,unique_image_ids_train,True,0 duplicate IDs found
4,schema_keys_test,True,missing: none
5,unique_image_ids_test,True,0 duplicate IDs found
6,no_invalid_or_missing_boxes,True,0 invalid/missing boxes remain
7,fix_0005014_excludes_box,True,
8,fix_0006857_excludes_box,True,
9,fix_0011654_excludes_box,True,


In [12]:
if n_failed == 0:
    print("ALL CHECKS PASSED — dataset is ready for fine-tuning.")
else:
    print(f"{n_failed} CHECK(S) FAILED — review before proceeding:")
    display(report_df[~report_df["passed"]])

ALL CHECKS PASSED — dataset is ready for fine-tuning.
